# dnabind playground

Load a trained model and predict binding for your own sequence pairs.

**Prerequisite:** train a model first (see the README quickstart), e.g.

```bash
cd data/demo && python make_demo_data.py && cd ../..
dnabind train --data_dir data/demo --encoder onehot_interleaved_reversed \
  --arch_config configs/cnn_4xn_small.json \
  --checkpoint experiments/demo/best_model.pt --log_dir experiments/demo --epochs 2 --eval
```

That produces `experiments/demo/best_model.pt`, which we load below.

In [2]:
from dnabind import load_model, list_encoders

# What encoders are available?
print("encoders:", list_encoders())

# Checkpoints are self-describing: they know their own encoder + sequence length.
CHECKPOINT = "experiments/demo/best_model.pt"  # adjust path if needed
model = load_model(CHECKPOINT)
print("loaded:", CHECKPOINT)

encoders: ['onehot_interleaved', 'wc_binary']
loaded: experiments/demo/best_model.pt


## Predict on a single pair

Both sequences are given 5'→3' and must be exactly the model's sequence length (20 by default).

In [3]:
seq1 = "AGCGATACGCCTTAACGTCT"
seq2 = "AATGGCGAAGGGGATCGTTC"

label, prob = model.predict(seq1, seq2)
print(f"seq1: {seq1}")
print(f"seq2: {seq2}")
print(f"prediction: {label}  (probability = {prob:.4f})")

seq1: AGCGATACGCCTTAACGTCT
seq2: AATGGCGAAGGGGATCGTTC
prediction: Bound  (probability = 0.7598)


## Sanity check: a perfect complement should read as strongly bound

The reverse complement of `seq1` forms a perfect antiparallel duplex, so the model should return a high probability.

In [4]:
complement = {"A": "T", "T": "A", "G": "C", "C": "G"}
revcomp = "".join(complement[b] for b in reversed(seq1))

label, prob = model.predict(seq1, revcomp)
print(f"seq1        : {seq1}")
print(f"revcomp(seq1): {revcomp}")
print(f"prediction  : {label}  (probability = {prob:.4f})")

seq1        : AGCGATACGCCTTAACGTCT
revcomp(seq1): AGACGTTAAGGCGTATCGCT
prediction  : Bound  (probability = 0.9999)


## Score a batch of pairs

Use `predict_proba` to get raw probabilities and rank candidates.

In [5]:
pairs = [
    ("AGCGATACGCCTTAACGTCT", "AATGGCGAAGGGGATCGTTC"),
    ("ACGTACGTACGTACGTACGT", "ACGTACGTACGTACGTACGT"),
    (seq1, revcomp),
]

for s1, s2 in sorted(pairs, key=lambda p: model.predict_proba(*p), reverse=True):
    p = model.predict_proba(s1, s2)
    print(f"{p:.4f}  {s1}  x  {s2}")

0.9999  AGCGATACGCCTTAACGTCT  x  AGACGTTAAGGCGTATCGCT
0.9990  ACGTACGTACGTACGTACGT  x  ACGTACGTACGTACGTACGT
0.7598  AGCGATACGCCTTAACGTCT  x  AATGGCGAAGGGGATCGTTC
